# Lab 8 — Memory-Backed Agent (with Stabilizers)
### *Add memory to an agent, then test when it helps vs hurts.*

<a href="https://colab.research.google.com/github/tulane-intro-ai-engineering/main/blob/main/labs/agents2_lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

---

## Overview
This lab adds **memory** to an agent:
- store notes (long-term memory)
- retrieve notes (embedding search)
- add stabilizers (budgets)

Small experiment:
> When does memory improve performance, and when does it introduce new failure modes?

---

## Learning goals
- Implement a tiny memory store (store + retrieve).
- Implement a minimal memory-aware agent.
- Add stabilizers: max calls / abstention.
- Run a controlled experiment: memory OFF vs ON.
- Build a Gradio demo to show behavior.


In [ ]:
# @title 🔧 Setup (Run this first)
!git clone --depth 1 -q https://github.com/tulane-intro-ai-engineering/main.git
import sys; sys.path.append("/content/main")

import re, json
import numpy as np
import pandas as pd

from course_utils import lab8_setup, get_text_embedding

lab8_setup()
import dspy
print("✅ Environment ready! dspy =", "yes" if dspy else "no")


installing mermaid-python
Enter your OpenAI API key. It will only live in this Colab runtime.
OpenAI API key: ··········
✅ API key set.
installing dspy
✅ Environment ready! dspy = yes


## Pre-Lab Questions
Answer in 1–2 sentences each. (Edit this cell.)

1. What is one benefit of long-term memory in an agent?
2. What is one risk of long-term memory?
3. Name one stabilizer that prevents runaway loops.

**Your answers:**
1)  
2)  
3)


## Scientific Question & Hypothesis

**Question:**  
If we change **X =** whether the agent uses memory (off vs on), what happens to **Y =** task success rate and failure rate?

**My hypothesis:**  
I expect memory will help on tasks that require _________, but hurt when _________.

Write your hypothesis here:


## Scientific process plan
- **Question:** Does memory improve performance on “remembering” tasks?
- **Hypothesis:** you wrote it above
- **Experiment:** run the same tasks with memory OFF vs ON
- **Measurement:** success rate + a simple “memory mistake rate”
- **Conclusion:** when should an agent store/retrieve memory?


# Part 1 — Tiny memory store (provided)
We’ll implement a simple memory store:
- `add(note)`
- `search(query, k)`

This uses embeddings, so retrieval feels similar to RAG.


In [ ]:
# @title TinyMemory (provided)
def normalize(v):
    v = np.array(v, dtype=np.float32)
    return v / (np.linalg.norm(v) + 1e-12)

class TinyMemory:
    def __init__(self):
        self.notes = []   # list of dicts: {"text":..., "tag":...}
        self.X = None     # embedding matrix

    def add(self, text, tag=""):
        self.notes.append({"text": text, "tag": tag})
        emb = normalize(get_text_embedding(text))
        self.X = emb[None, :] if self.X is None else np.vstack([self.X, emb])

    def search(self, query, k=3):
        if self.X is None:
            return []
        q = normalize(get_text_embedding(query))
        sims = self.X @ q
        idx = np.argsort(-sims)[:k]
        return [self.notes[int(i)] for i in idx]

mem = TinyMemory()
mem.add("User likes short bullet answers.", tag="pref")
mem.add("Project Bluebird deadline is Feb 1.", tag="project")
mem.add("Interns may join on-call only after manager approval.", tag="policy")
mem.search("When is Bluebird due?", k=2)


[{'text': 'Project Bluebird deadline is Feb 1.', 'tag': 'project'},
 {'text': 'User likes short bullet answers.', 'tag': 'pref'}]

# Part 2 — Build a minimal memory-aware agent (TODOs)
We’ll keep the agent simple:

- It sees a **question**
- It may retrieve **top-k memory notes**
- It decides whether to use memory
- It produces a short answer

You’ll implement:
- `should_use_memory(question)`
- `format_memory(notes)`
- `agent_answer(question, use_memory=True)`


In [ ]:
# @title ✅ TODO: Implement memory policy + formatting
def should_use_memory(question: str) -> bool:
    # TODO:
    # Return True if the question likely benefits from memory.
    # Hint keywords: 'my', 'we', 'our', 'remember', 'preference', 'project', 'deadline'
    raise NotImplementedError("Implement should_use_memory(question)")

def format_memory(notes) -> str:
    # TODO:
    # Convert list of notes into a short string (1 note per line).
    raise NotImplementedError("Implement format_memory(notes)")


In [ ]:
# @title minimal model call in DSPy
lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)
print("DSPy LM configured ✅")

def fallback_answer(question, memory_text=""):
    if memory_text:
        return "I found these notes that might help:\n" + memory_text
    return "No model configured. (Set OPENAI_API_KEY to generate real answers.)"

def dspy_answer(question, memory_text=""):
    # Keep the prompt simple.
    prompt = "Answer the question. If MEMORY is provided, use it.\n\n"
    if memory_text:
        prompt += "MEMORY:\n" + memory_text + "\n\n"
    prompt += "QUESTION: " + question + "\nANSWER:"
    return lm(prompt)  # if lm configured


In [ ]:
# @title ✅ TODO: Implement agent_answer(question, use_memory=True)
def agent_answer(question: str, use_memory: bool = True, k_notes: int = 3) -> dict:
    # Return dict with keys:
    #   question, used_memory, retrieved_notes, answer
    #
    # If use_memory is True:
    #   - compute want = should_use_memory(question)
    #   - if want: notes = mem.search(question, k=k_notes)
    #   - memory_text = format_memory(notes)
    # Else: notes=[], memory_text=""
    #
    # Answer:
    #   - if lm configured: answer = dspy_answer(question, memory_text)
    #   - else: answer = fallback_answer(question, memory_text)
    #
    # TODO: implement.
    raise NotImplementedError("Implement agent_answer(...)")


# Part 3 — Experiment: Memory OFF vs ON
We’ll test on a small task set and measure success rate.

We’ll use a very lightweight success check:
- if a task has a required substring, success means the answer contains it
- otherwise, success means you produced a non-empty answer


In [ ]:
# @title Task set (provided)
tasks = [
    {"q": "When is Project Bluebird due?", "gold_contains": "Feb"},
    {"q": "How do I like answers formatted?", "gold_contains": "bullet"},
    {"q": "What is an agent?", "gold_contains": None},  # should not need memory
    {"q": "Summarize what RAG is.", "gold_contains": None},
]

def simple_success(task, answer_text):
    if task["gold_contains"] is None:
        return True if (answer_text and answer_text.strip()) else False
    return task["gold_contains"].lower() in answer_text.lower()

tasks


In [ ]:
# @title ✅ TODO: Run experiment and log results
def run_experiment(use_memory: bool):
    logs=[]
    for t in tasks:
        out = agent_answer(t["q"], use_memory=use_memory, k_notes=3)
        ans = out["answer"]
        ok = simple_success(t, ans)
        logs.append({
            "question": t["q"],
            "use_memory_setting": use_memory,
            "used_memory": out["used_memory"],
            "success": ok,
            "answer_preview": ans[:120] + "…",
        })
    return pd.DataFrame(logs)

df_off = run_experiment(use_memory=False)
df_on  = run_experiment(use_memory=True)

print("Success rate (memory OFF):", df_off["success"].mean())
print("Success rate (memory ON): ", df_on["success"].mean())

display(df_on)


### Reflection
- Did memory help on “remembering” tasks?
- Did it get used on questions that didn’t need it?

Write here:


# Part 4 — Stabilizers: budgets + abstention
Even with a one-step agent, get into the habit of budgets.

You’ll implement a wrapper:
- if max_calls < 1 → abstain
- else → call agent_answer once

This seems small now, but it becomes critical for multi-step agents.


In [ ]:
# @title ✅ TODO: Add a simple budget wrapper
def budgeted_agent_answer(question: str, max_calls: int = 1, use_memory: bool = True) -> dict:
    # If max_calls < 1:
    #   return {"question": question, "answer": "I don't know (budget exceeded).", ...}
    # Otherwise:
    #   return agent_answer(question, use_memory=use_memory)
    raise NotImplementedError("Implement budgeted_agent_answer(...)")


# Part 5 (Optional) — Gradio demo
Show your memory agent:
- toggle memory on/off
- show retrieved notes
- show answer


In [ ]:
# @title Optional: Gradio demo (minimal)
# !pip -q install gradio
import gradio as gr

def ui_run(question, memory_on):
    out = agent_answer(question, use_memory=memory_on, k_notes=3)
    notes = "\n".join("- " + n["text"] for n in out["retrieved_notes"]) if out["retrieved_notes"] else "(none)"
    return out["used_memory"], notes, out["answer"]

demo = gr.Interface(
    fn=ui_run,
    inputs=[
        gr.Textbox(label="Question", value="When is Project Bluebird due?"),
        gr.Checkbox(label="Memory enabled", value=True),
    ],
    outputs=[
        gr.Checkbox(label="Agent used memory?"),
        gr.Textbox(label="Retrieved notes"),
        gr.Textbox(label="Answer"),
    ],
    title="Lab 8: Memory-Backed Agent (minimal)",
)
demo.launch(debug=False, share=False)


---

## Results
Summarize what you observed:
- success rate OFF vs ON
- one example where memory helped
- one example where memory was irrelevant or risky

Write here:


## Conclusion
- Was your hypothesis supported?
- When should an agent store/retrieve memory?
- What guardrail would you add to prevent storing unsafe info?

Write here:


## Post-Lab Reflection
Answer briefly (2–4 sentences each). (Edit this cell.)

1. What was the most surprising failure mode you saw?
2. If you shipped memory in a product, what would you log/monitor?
3. What do you want Agents III to cover?

Your answers:
1)  
2)  
3)


---

## 🧠 AI Usage Log

> Use this section to document any generative AI assistance (e.g., ChatGPT, Claude, Copilot) you used while completing this lab or assignment.  
> Be specific — transparency and reflection matter more than the amount of AI use.


| Tool Used | Purpose | Prompt / Context | Verification & Edits |
|------------|----------|------------------|----------------------|
| (e.g., ChatGPT (GPT-5)) | (e.g., debugging, code explanation, idea generation) | (e.g., "Why does my cosine similarity return NaN?") | (e.g., ran tests on sample input, compared with lecture code) |
| (Add rows as needed) | | | |

**Summary (2–3 sentences):**  
Briefly describe what you learned or how AI helped you think through the problem.  
Example: *AI helped me notice an off-by-one error in my indexing. I double-checked by printing intermediate results and confirmed the fix.*

---



In [ ]:
# @title ✅ Checks for Lab 8
print("Running checks...")

try:
    v = should_use_memory("remember my preference")
    assert isinstance(v, bool)
    print("✅ should_use_memory returns bool.")
except Exception as e:
    print("❌ should_use_memory check failed:", e)

try:
    s = format_memory([{"text":"a","tag":""}])
    assert isinstance(s, str)
    print("✅ format_memory returns str.")
except Exception as e:
    print("❌ format_memory check failed:", e)

try:
    out = agent_answer("When is Bluebird due?", use_memory=True)
    for k in ["question","used_memory","retrieved_notes","answer"]:
        assert k in out
    print("✅ agent_answer output format ok.")
except Exception as e:
    print("❌ agent_answer check failed:", e)

try:
    out = budgeted_agent_answer("test", max_calls=0)
    assert isinstance(out, dict) and "answer" in out
    print("✅ budgeted_agent_answer basic behavior ok.")
except Exception as e:
    print("❌ budgeted_agent_answer check failed:", e)

print("Done.")
